In [3]:
# ============================================================
# PHASE 7A — ESSENTIAL ANALYTICS ENGINEERING MODELS
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# Build the essential Analytics Engineering layer on top of
# the trusted Phase 6 MySQL architecture.
#
# This phase focuses ONLY on:
#
#   1. Reusable analytical models
#   2. Explicit model grain
#   3. Safe customer-dimension handling
#   4. Core working-capital calculations
#   5. Essential automated tests
#   6. Financial reconciliation
#   7. Model inventory
#
# We intentionally DO NOT build:
#   - metadata framework
#   - AE run tables
#   - lineage tables
#   - large Python hashing framework
#   - 20+ automated tests
#   - CI/CD
#
# Those concepts can be added later if they materially improve
# the portfolio.
#
# IMPORTANT:
# Phase 6 data is NOT modified.
#
# Source:
#     core.fact_invoice
#     core.dim_customer
#     core.dim_supplier
#     core.dim_product
#
# Target:
#     analytics.ae_fact_working_capital
#     analytics.ae_customer_quality
#     analytics.ae_ar_invoice
#     analytics.ae_payment_behavior
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import mysql.connector
import pandas as pd
import time
from datetime import datetime
import os

# ============================================================
# 2. MYSQL CONFIGURATION
# ============================================================

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "working_capital")
}



# ============================================================
# 3. START
# ============================================================

print("=" * 70)
print("PHASE 7A — ESSENTIAL ANALYTICS ENGINEERING")
print("=" * 70)

start_time = time.time()


# ============================================================
# 4. MYSQL CONNECTION
# ============================================================

conn = mysql.connector.connect(**MYSQL_CONFIG)
cursor = conn.cursor()

print("\nMYSQL connection: PASS")


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def execute_sql(sql, params=None):
    cursor.execute(sql, params or ())
    conn.commit()


def scalar_query(sql, params=None):
    cursor.execute(sql, params or ())
    result = cursor.fetchone()

    if result is None:
        return None

    return result[0]


def fetch_all(sql, params=None):
    cursor.execute(sql, params or ())
    return cursor.fetchall()


def test_result(test_name, expected, actual, passed):
    status = "PASS" if passed else "FAIL"

    print(
        f"{status:<6} | "
        f"{test_name:<45} | "
        f"expected={expected} | "
        f"actual={actual}"
    )

    return passed


# ============================================================
# 6. VALIDATE PHASE 6 SOURCE
# ============================================================

print("\n1. VALIDATE PHASE 6 SOURCE")


core_rows = scalar_query("""
    SELECT COUNT(*)
    FROM core.fact_invoice
""")


core_unique_ids = scalar_query("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM core.fact_invoice
""")


print(f"core.fact_invoice rows   : {core_rows:,}")
print(f"core unique invoice IDs : {core_unique_ids:,}")


if core_rows != 12000:
    raise ValueError(
        f"Phase 6 source failure: expected 12,000 rows, got {core_rows}"
    )


if core_unique_ids != 12000:
    raise ValueError(
        f"Phase 6 source failure: expected 12,000 unique invoice IDs, "
        f"got {core_unique_ids}"
    )


print("Phase 6 source validation: PASS")


# ============================================================
# 7. RESET ONLY PHASE 7A VIEWS
# ============================================================

print("\n2. RESET PHASE 7A VIEWS")


# Drop dependent model first.

execute_sql("""
    DROP VIEW IF EXISTS analytics.ae_ar_invoice
""")


execute_sql("""
    DROP VIEW IF EXISTS analytics.ae_payment_behavior
""")


execute_sql("""
    DROP VIEW IF EXISTS analytics.ae_customer_quality
""")


execute_sql("""
    DROP VIEW IF EXISTS analytics.ae_fact_working_capital
""")


print("Phase 7A views reset: PASS")


# ============================================================
# 8. CREATE AE FACT MODEL
# ============================================================
#
# GRAIN:
#     1 row = 1 invoice
#
# Source:
#     core.fact_invoice
#
# This is the main reusable analytical foundation.
#
# ============================================================

print("\n3. CREATE AE FACT MODEL")


execute_sql("""
CREATE VIEW analytics.ae_fact_working_capital AS

SELECT

    f.invoice_id,

    f.customer_id,

    f.supplier_id,

    f.product_id,

    f.invoice_date,

    f.due_date,

    f.payment_date,

    f.invoice_amount,

    f.payment_amount,

    f.currency,

    f.payment_method,

    f.status,

    f.business_rule_status,

    f.financial_analytical_eligibility,

    f.days_to_payment,

    f.days_from_due_date,


    /* --------------------------------------------------------
       PAYMENT TIMING
       -------------------------------------------------------- */

    CASE
        WHEN f.payment_date IS NOT NULL
         AND f.due_date IS NOT NULL
         AND f.payment_date < f.due_date

        THEN DATEDIFF(
            f.due_date,
            f.payment_date
        )

        ELSE 0
    END AS days_paid_early,


    CASE
        WHEN f.payment_date IS NOT NULL
         AND f.due_date IS NOT NULL
         AND f.payment_date > f.due_date

        THEN DATEDIFF(
            f.payment_date,
            f.due_date
        )

        ELSE 0
    END AS days_paid_late,


    CASE
        WHEN f.payment_date IS NOT NULL
         AND f.due_date IS NOT NULL
         AND f.payment_date > f.due_date

        THEN 1

        ELSE 0
    END AS is_paid_late,


    /* --------------------------------------------------------
       STATUS FLAGS
       -------------------------------------------------------- */

    CASE
        WHEN f.status = 'Paid'
        THEN 1
        ELSE 0
    END AS is_paid,


    CASE
        WHEN f.status = 'Open'
        THEN 1
        ELSE 0
    END AS is_open_invoice,


    /* --------------------------------------------------------
       ANALYTICAL ELIGIBILITY
       -------------------------------------------------------- */

    CASE
        WHEN f.financial_analytical_eligibility = 'FULL'
        THEN 1
        ELSE 0
    END AS is_amount_analysis_eligible,


    /* --------------------------------------------------------
       PAYMENT VARIANCE
       -------------------------------------------------------- */

    CASE
        WHEN f.invoice_amount IS NOT NULL
         AND f.payment_amount IS NOT NULL

        THEN
            f.payment_amount - f.invoice_amount

        ELSE NULL
    END AS payment_variance,


    /* --------------------------------------------------------
       PAYMENT COMPLETION RATIO
       -------------------------------------------------------- */

    CASE
        WHEN f.invoice_amount IS NOT NULL
         AND f.invoice_amount <> 0
         AND f.payment_amount IS NOT NULL

        THEN
            f.payment_amount / f.invoice_amount

        ELSE NULL
    END AS payment_completion_ratio


FROM core.fact_invoice f
""")


print("analytics.ae_fact_working_capital: CREATED")


# ============================================================
# 9. CREATE CUSTOMER QUALITY MODEL
# ============================================================
#
# IMPORTANT:
#
# core.dim_customer does NOT have one row per customer_id.
#
# Therefore we NEVER directly join:
#
#     fact_invoice -> dim_customer
#
# Instead we first create:
#
#     1 row = 1 customer_id
#
# and classify whether that customer ID is unambiguous.
#
# ============================================================

print("\n4. CREATE CUSTOMER QUALITY MODEL")


execute_sql("""
CREATE VIEW analytics.ae_customer_quality AS

SELECT

    customer_id,

    COUNT(*) AS dimension_row_count,

    COUNT(DISTINCT customer_name)
        AS distinct_customer_names,

    COUNT(DISTINCT region)
        AS distinct_regions,

    CASE

        WHEN COUNT(*) = 1
         AND COUNT(DISTINCT customer_name) = 1
         AND COUNT(DISTINCT region) = 1

        THEN 'UNAMBIGUOUS'

        ELSE 'AMBIGUOUS'

    END AS customer_dimension_status


FROM core.dim_customer


WHERE customer_id IS NOT NULL


GROUP BY customer_id
""")


customer_quality_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_customer_quality
""")


customer_ambiguous = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_customer_quality
    WHERE customer_dimension_status = 'AMBIGUOUS'
""")


print(
    f"Customer IDs represented : {customer_quality_rows:,}"
)

print(
    f"Ambiguous customer IDs   : {customer_ambiguous:,}"
)

print("analytics.ae_customer_quality: CREATED")


# ============================================================
# 10. CREATE AR INVOICE MODEL
# ============================================================
#
# GRAIN:
#     1 row = 1 invoice
#
# Customer quality is joined only after aggregation to one
# row per customer_id.
#
# This prevents invoice row multiplication.
#
# ============================================================

print("\n5. CREATE AR INVOICE MODEL")


execute_sql("""
CREATE VIEW analytics.ae_ar_invoice AS

SELECT

    f.invoice_id,

    f.customer_id,

    f.supplier_id,

    f.product_id,

    f.invoice_date,

    f.due_date,

    f.payment_date,

    f.invoice_amount,

    f.payment_amount,

    f.currency,

    f.payment_method,

    f.status,

    f.business_rule_status,

    f.financial_analytical_eligibility,

    f.days_to_payment,

    f.days_from_due_date,

    f.days_paid_early,

    f.days_paid_late,

    f.is_paid_late,

    f.is_paid,

    f.is_open_invoice,

    f.is_amount_analysis_eligible,

    f.payment_variance,

    f.payment_completion_ratio,


    /* --------------------------------------------------------
       CUSTOMER QUALITY
       -------------------------------------------------------- */

    COALESCE(
        cq.dimension_row_count,
        0
    ) AS customer_dimension_row_count,


    COALESCE(
        cq.distinct_customer_names,
        0
    ) AS customer_distinct_name_count,


    COALESCE(
        cq.distinct_regions,
        0
    ) AS customer_distinct_region_count,


    COALESCE(
        cq.customer_dimension_status,
        'NO_CUSTOMER_DIMENSION_RECORD'
    ) AS customer_dimension_status,


    /* --------------------------------------------------------
       CURRENT OVERDUE STATUS
       --------------------------------------------------------
       This is intentionally dynamic.
       It will be used later for operational/current AR analysis.
       -------------------------------------------------------- */

    CASE

        WHEN f.status = 'Open'
         AND f.due_date IS NOT NULL
         AND f.due_date < CURRENT_DATE()

        THEN 1

        ELSE 0

    END AS currently_overdue,


    CASE

        WHEN f.status = 'Open'
         AND f.due_date IS NOT NULL
         AND f.due_date < CURRENT_DATE()
         AND f.invoice_amount IS NOT NULL

        THEN f.invoice_amount

        ELSE 0

    END AS currently_overdue_amount


FROM analytics.ae_fact_working_capital f


LEFT JOIN analytics.ae_customer_quality cq

    ON f.customer_id = cq.customer_id
""")


print("analytics.ae_ar_invoice: CREATED")


# ============================================================
# 11. CREATE PAYMENT BEHAVIOR MODEL
# ============================================================
#
# GRAIN:
#     1 row = currency + payment_method + status
#
# ============================================================

print("\n6. CREATE PAYMENT BEHAVIOR MODEL")


execute_sql("""
CREATE VIEW analytics.ae_payment_behavior AS

SELECT

    currency,

    payment_method,

    status,

    COUNT(*) AS invoice_count,


    SUM(
        CASE
            WHEN invoice_amount IS NOT NULL
            THEN invoice_amount
            ELSE 0
        END
    ) AS invoice_amount_total,


    SUM(
        CASE
            WHEN payment_amount IS NOT NULL
            THEN payment_amount
            ELSE 0
        END
    ) AS payment_amount_total,


    AVG(
        CASE
            WHEN days_to_payment IS NOT NULL
            THEN days_to_payment
            ELSE NULL
        END
    ) AS average_days_to_payment,


    SUM(
        CASE
            WHEN is_paid_late = 1
            THEN 1
            ELSE 0
        END
    ) AS late_payment_count,


    SUM(
        CASE
            WHEN is_open_invoice = 1
            THEN 1
            ELSE 0
        END
    ) AS open_invoice_count,


    SUM(
        CASE
            WHEN is_amount_analysis_eligible = 1
            THEN 1
            ELSE 0
        END
    ) AS amount_analysis_eligible_count


FROM analytics.ae_fact_working_capital


GROUP BY

    currency,

    payment_method,

    status
""")


print("analytics.ae_payment_behavior: CREATED")


# ============================================================
# 12. BASIC MODEL INVENTORY
# ============================================================

print("\n7. ANALYTICS MODEL INVENTORY")


model_inventory = fetch_all("""
SELECT
    TABLE_NAME,
    TABLE_TYPE
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'analytics'
ORDER BY TABLE_NAME
""")


for table_name, table_type in model_inventory:

    print(
        f"analytics.{table_name:<40} {table_type}"
    )


# ============================================================
# 13. AUTOMATED TESTS
# ============================================================
#
# Small and targeted.
#
# These tests verify the things that actually matter for the
# analytical layer.
#
# ============================================================

print("\n8. ESSENTIAL ANALYTICS ENGINEERING TESTS")


tests = []


# ------------------------------------------------------------
# TEST 1 — FACT ROW COUNT
# ------------------------------------------------------------

ae_fact_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
""")


tests.append(
    test_result(
        "AE fact row count",
        12000,
        ae_fact_rows,
        ae_fact_rows == 12000
    )
)


# ------------------------------------------------------------
# TEST 2 — FACT UNIQUE INVOICE IDS
# ------------------------------------------------------------

ae_fact_unique_ids = scalar_query("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM analytics.ae_fact_working_capital
""")


tests.append(
    test_result(
        "AE fact invoice uniqueness",
        12000,
        ae_fact_unique_ids,
        ae_fact_unique_ids == 12000
    )
)


# ------------------------------------------------------------
# TEST 3 — AR ROW COUNT
# ------------------------------------------------------------

ar_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_ar_invoice
""")


tests.append(
    test_result(
        "AR model row count",
        12000,
        ar_rows,
        ar_rows == 12000
    )
)


# ------------------------------------------------------------
# TEST 4 — AR INVOICE UNIQUENESS
# ------------------------------------------------------------

ar_unique_ids = scalar_query("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM analytics.ae_ar_invoice
""")


tests.append(
    test_result(
        "AR model invoice uniqueness",
        12000,
        ar_unique_ids,
        ar_unique_ids == 12000
    )
)


# ------------------------------------------------------------
# TEST 5 — NO ROW MULTIPLICATION
# ------------------------------------------------------------

duplicate_ar_ids = scalar_query("""
    SELECT COUNT(*)
    FROM (
        SELECT invoice_id
        FROM analytics.ae_ar_invoice
        GROUP BY invoice_id
        HAVING COUNT(*) > 1
    ) x
""")


tests.append(
    test_result(
        "No invoice row multiplication",
        0,
        duplicate_ar_ids,
        duplicate_ar_ids == 0
    )
)


# ------------------------------------------------------------
# TEST 6 — FINANCIAL RECONCILIATION: INVOICE AMOUNT
# ------------------------------------------------------------

core_invoice_total = scalar_query("""
    SELECT COALESCE(SUM(invoice_amount), 0)
    FROM core.fact_invoice
""")


ae_invoice_total = scalar_query("""
    SELECT COALESCE(SUM(invoice_amount), 0)
    FROM analytics.ae_fact_working_capital
""")


invoice_difference = float(
    ae_invoice_total - core_invoice_total
)


tests.append(
    test_result(
        "Invoice amount reconciliation",
        "difference = 0",
        f"{invoice_difference:.6f}",
        abs(invoice_difference) < 0.01
    )
)


# ------------------------------------------------------------
# TEST 7 — FINANCIAL RECONCILIATION: PAYMENT AMOUNT
# ------------------------------------------------------------

core_payment_total = scalar_query("""
    SELECT COALESCE(SUM(payment_amount), 0)
    FROM core.fact_invoice
""")


ae_payment_total = scalar_query("""
    SELECT COALESCE(SUM(payment_amount), 0)
    FROM analytics.ae_fact_working_capital
""")


payment_difference = float(
    ae_payment_total - core_payment_total
)


tests.append(
    test_result(
        "Payment amount reconciliation",
        "difference = 0",
        f"{payment_difference:.6f}",
        abs(payment_difference) < 0.01
    )
)


# ------------------------------------------------------------
# TEST 8 — STATUS RECONCILIATION
# ------------------------------------------------------------

core_status = dict(
    fetch_all("""
        SELECT status, COUNT(*)
        FROM core.fact_invoice
        GROUP BY status
    """)
)


ae_status = dict(
    fetch_all("""
        SELECT status, COUNT(*)
        FROM analytics.ae_fact_working_capital
        GROUP BY status
    """)
)


tests.append(
    test_result(
        "Status distribution reconciliation",
        core_status,
        ae_status,
        core_status == ae_status
    )
)


# ------------------------------------------------------------
# TEST 9 — CURRENCY RECONCILIATION
# ------------------------------------------------------------

core_currency = dict(
    fetch_all("""
        SELECT currency, COUNT(*)
        FROM core.fact_invoice
        GROUP BY currency
    """)
)


ae_currency = dict(
    fetch_all("""
        SELECT currency, COUNT(*)
        FROM analytics.ae_fact_working_capital
        GROUP BY currency
    """)
)


tests.append(
    test_result(
        "Currency distribution reconciliation",
        core_currency,
        ae_currency,
        core_currency == ae_currency
    )
)


# ------------------------------------------------------------
# TEST 10 — PAYMENT DATE BEFORE INVOICE
# ------------------------------------------------------------

payment_before_invoice = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
    WHERE payment_date IS NOT NULL
      AND invoice_date IS NOT NULL
      AND payment_date < invoice_date
""")


tests.append(
    test_result(
        "Payment date >= invoice date",
        0,
        payment_before_invoice,
        payment_before_invoice == 0
    )
)


# ------------------------------------------------------------
# TEST 11 — DUE DATE BEFORE INVOICE
# ------------------------------------------------------------

due_before_invoice = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
    WHERE due_date IS NOT NULL
      AND invoice_date IS NOT NULL
      AND due_date < invoice_date
""")


tests.append(
    test_result(
        "Due date >= invoice date",
        0,
        due_before_invoice,
        due_before_invoice == 0
    )
)


# ------------------------------------------------------------
# TEST 12 — CUSTOMER QUALITY GRAIN
# ------------------------------------------------------------

customer_quality_duplicates = scalar_query("""
    SELECT COUNT(*)
    FROM (
        SELECT customer_id
        FROM analytics.ae_customer_quality
        GROUP BY customer_id
        HAVING COUNT(*) > 1
    ) x
""")


tests.append(
    test_result(
        "Customer quality one row per customer",
        0,
        customer_quality_duplicates,
        customer_quality_duplicates == 0
    )
)


# ------------------------------------------------------------
# TEST 13 — CUSTOMER JOIN PRESERVES INVOICE GRAIN
# ------------------------------------------------------------

customer_join_duplicates = scalar_query("""
    SELECT COUNT(*)
    FROM (
        SELECT invoice_id
        FROM analytics.ae_ar_invoice
        GROUP BY invoice_id
        HAVING COUNT(*) > 1
    ) x
""")


tests.append(
    test_result(
        "Customer join preserves invoice grain",
        0,
        customer_join_duplicates,
        customer_join_duplicates == 0
    )
)


# ------------------------------------------------------------
# TEST 14 — PAYMENT VARIANCE LOGIC
# ------------------------------------------------------------

variance_errors = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
    WHERE
        (
            invoice_amount IS NOT NULL
            AND payment_amount IS NOT NULL
            AND (
                payment_variance IS NULL
                OR ABS(
                    payment_variance
                    - (payment_amount - invoice_amount)
                ) > 0.01
            )
        )
        OR
        (
            (
                invoice_amount IS NULL
                OR payment_amount IS NULL
            )
            AND payment_variance IS NOT NULL
        )
""")


tests.append(
    test_result(
        "Payment variance logic",
        0,
        variance_errors,
        variance_errors == 0
    )
)


# ------------------------------------------------------------
# TEST 15 — LATE PAYMENT LOGIC
# ------------------------------------------------------------

late_payment_errors = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
    WHERE
        (
            payment_date IS NOT NULL
            AND due_date IS NOT NULL
            AND payment_date > due_date
            AND is_paid_late <> 1
        )
        OR
        (
            (
                payment_date IS NULL
                OR due_date IS NULL
                OR payment_date <= due_date
            )
            AND is_paid_late <> 0
        )
""")


tests.append(
    test_result(
        "Late payment flag logic",
        0,
        late_payment_errors,
        late_payment_errors == 0
    )
)


# ============================================================
# 14. FINAL TEST SUMMARY
# ============================================================

print("\n9. FINAL TEST SUMMARY")


total_tests = len(tests)

passed_tests = sum(
    1 for result in tests
    if result
)

failed_tests = total_tests - passed_tests


print(f"Total tests : {total_tests}")
print(f"Passed      : {passed_tests}")
print(f"Failed      : {failed_tests}")


if failed_tests > 0:

    print("\nPHASE 7A TEST FAILURE")

    cursor.close()
    conn.close()

    raise ValueError(
        f"Phase 7A failed: {failed_tests} test(s) failed."
    )


# ============================================================
# 15. FINAL MODEL COUNTS
# ============================================================

print("\n10. FINAL MODEL RECONCILIATION")


final_fact_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_fact_working_capital
""")


final_ar_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_ar_invoice
""")


final_customer_quality_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_customer_quality
""")


final_payment_behavior_rows = scalar_query("""
    SELECT COUNT(*)
    FROM analytics.ae_payment_behavior
""")


print(
    f"AE fact working capital : {final_fact_rows:,}"
)

print(
    f"AE AR invoice           : {final_ar_rows:,}"
)

print(
    f"AE customer quality     : {final_customer_quality_rows:,}"
)

print(
    f"AE payment behavior     : {final_payment_behavior_rows:,}"
)


# ============================================================
# 16. SAVE SMALL PHASE 7A REPORT
# ============================================================

print("\n11. SAVE PHASE 7A REPORT")


report_path = (
    r"C:\Users\hp\Downloads\Working_Capital_ETL"
    r"\Phase7A_Analytics_Engineering_Report.csv"
)


report_rows = [

    {
        "metric": "phase",
        "value": "PHASE_7A"
    },

    {
        "metric": "status",
        "value": "SUCCESS"
    },

    {
        "metric": "total_tests",
        "value": total_tests
    },

    {
        "metric": "passed_tests",
        "value": passed_tests
    },

    {
        "metric": "failed_tests",
        "value": failed_tests
    },

    {
        "metric": "core_fact_rows",
        "value": core_rows
    },

    {
        "metric": "ae_fact_rows",
        "value": final_fact_rows
    },

    {
        "metric": "ae_ar_invoice_rows",
        "value": final_ar_rows
    },

    {
        "metric": "customer_quality_rows",
        "value": final_customer_quality_rows
    },

    {
        "metric": "ambiguous_customer_ids",
        "value": customer_ambiguous
    },

    {
        "metric": "payment_behavior_rows",
        "value": final_payment_behavior_rows
    },

    {
        "metric": "invoice_amount_difference",
        "value": invoice_difference
    },

    {
        "metric": "payment_amount_difference",
        "value": payment_difference
    }
]


report_df = pd.DataFrame(report_rows)


report_df.to_csv(
    report_path,
    index=False
)


print(
    f"Report saved: {report_path}"
)


# ============================================================
# 17. CLOSE CONNECTION
# ============================================================

cursor.close()
conn.close()


elapsed_time = time.time() - start_time


# ============================================================
# 18. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("PHASE 7A FINAL STATUS")
print("=" * 70)

print("Pipeline status : SUCCESS")
print(f"Tests passed    : {passed_tests}/{total_tests}")
print(f"Tests failed    : {failed_tests}")
print(f"AE fact rows    : {final_fact_rows:,}")
print(f"AR invoice rows : {final_ar_rows:,}")
print(f"Elapsed time    : {elapsed_time:.2f} seconds")

print("=" * 70)

print("PHASE 7A COMPLETE — SUCCESS")

PHASE 7A — ESSENTIAL ANALYTICS ENGINEERING

MYSQL connection: PASS

1. VALIDATE PHASE 6 SOURCE
core.fact_invoice rows   : 12,000
core unique invoice IDs : 12,000
Phase 6 source validation: PASS

2. RESET PHASE 7A VIEWS
Phase 7A views reset: PASS

3. CREATE AE FACT MODEL
analytics.ae_fact_working_capital: CREATED

4. CREATE CUSTOMER QUALITY MODEL
Customer IDs represented : 80
Ambiguous customer IDs   : 80
analytics.ae_customer_quality: CREATED

5. CREATE AR INVOICE MODEL
analytics.ae_ar_invoice: CREATED

6. CREATE PAYMENT BEHAVIOR MODEL
analytics.ae_payment_behavior: CREATED

7. ANALYTICS MODEL INVENTORY
analytics.ae_ar_invoice                            VIEW
analytics.ae_customer_quality                      VIEW
analytics.ae_fact_working_capital                  VIEW
analytics.ae_payment_behavior                      VIEW
analytics.fact_working_capital                     BASE TABLE

8. ESSENTIAL ANALYTICS ENGINEERING TESTS
PASS   | AE fact row count                             | expe